In [1]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go


Data From CENTER OF AMERICAN WOMEN AND POLITICS (CAWP) - RUTGERS UNIVERSITY, MIT ELECTION LAB AND HARVARD DATAVERSE

Women congress: https://cawp.rutgers.edu/candidates-2026-election-results-tracker/women-120th-congress

Current women: https://cawp.rutgers.edu/data/women-elected-officials-database?tab=ByParty

Candidates: https://cawp.rutgers.edu/data/candidates-women-candidates-2026?tab=CandidateList

Rebound candidates: https://cawp.rutgers.edu/data/candidates-women-candidates-2026?tab=ReboundCandidates

Senate 2022: https://dataverse.harvard.edu/file.xhtml?fileId=7412054&version=3.0

women_candidates_2024: https://cawp.rutgers.edu/data/candidates-election-results-and-analysis-0/2024-summary-women-candidates

https://electionlab.mit.edu/data

1974-2024 House: https://dataverse.harvard.edu/ 

In [2]:
data_folder = Path("../data").resolve()

In [3]:
women_versus_women = pd.read_csv(data_folder / '2026 Woman V Woman.csv')
women_versus_women[["State", "Office"]] = women_versus_women[["State", "Office"]].ffill()

In [4]:
women_rebound = pd.read_csv(data_folder / '2026 Rebound Candidates.csv')

In [5]:
current_women = pd.read_csv(data_folder / 'search_officeholders-05112026.csv')

In [6]:
women_congress = pd.read_excel(data_folder / '2026 Women congress.xlsx', sheet_name='Data')

In [7]:
senate_2022 = pd.read_csv(data_folder / 'senate_2022.csv')

In [8]:
historical_representatives = pd.read_csv(data_folder / '1976-2024-house.tab', sep=',', dtype={'runoff': 'string'})
historical_representatives['candidate'] = (historical_representatives['candidate'].str.replace(r'\"', '"', regex=False))

In [9]:
representatives_2024 = historical_representatives[historical_representatives['year'] == 2024]

In [10]:
representatives_2024_clean = representatives_2024[(~representatives_2024['writein'].fillna(False))&~(representatives_2024['candidate'].str.upper()=='WRITEIN')]
representatives_2024_clean_win = (representatives_2024.sort_values('candidatevotes', ascending=False).drop_duplicates(subset=['state', 'district'], keep='first'))

In [11]:
df_2024=representatives_2024_clean_win.drop_duplicates(subset=['office', 'state_po', 'candidate', 'party']).groupby(['office', 'state_po','party']).size().reset_index(name='Total_2024')
df_2024['office'] = df_2024['office'].replace({'US HOUSE': 'U.S. Representative'})
df_2024['party'] = df_2024['party'].replace({'REPUBLICAN': 'R', 'DEMOCRAT': 'D'})
df_2024=df_2024.rename(columns={'state_po': 'State', 'office': 'Office', 'party': 'Party'})

In [12]:
women_candidates_2024 = pd.read_excel(data_folder / 'women_candidates_2024.xlsx', sheet_name='Data')
women_candidates_2024 = women_candidates_2024[women_candidates_2024['Office'].isin(['U.S. Rep.', 'U.S. Sen.'])]
women_candidates_2024 = women_candidates_2024[women_candidates_2024['General Election Result']=='W']

In [13]:
df_w_2026=women_congress.drop_duplicates(subset=[ 'Office', 'State','First Name', 'Last Name', 'Party']).groupby(['Office','State', 'Party']).size().reset_index(name='Num_Women')

In [14]:
df_w_2024=women_candidates_2024.drop_duplicates(subset=[ 'Office', 'State','First Name', 'Last Name', 'Party']).groupby(['Office','State', 'Party']).size().reset_index(name='Num_Women')
df_w_2024['Office'] = df_w_2024['Office'].replace({'U.S. Rep.' :'U.S. Representative', 'U.S. Sen.': 'U.S. Senator'})
df_w_2024_complete = df_2024.merge(df_w_2024, on=['Office', 'State', 'Party'], how='left', suffixes=('_total', '_women'))


In [15]:
df_comparaison=df_w_2026[(df_w_2026['Office']=='U.S. Representative')].merge(df_w_2024_complete[~df_w_2024_complete['State'].eq('DC')], on=['Office', 'State', 'Party'], how='outer', suffixes=('_2026', '_2024')).fillna({'Num_Women_2026': 0, 'Total_2024': 0, 'Num_Women_2024': 0})

In [16]:
df_comparaison ['diff']= df_comparaison['Num_Women_2024']-df_comparaison['Num_Women_2026']

In [17]:
df_comparaison_women = df_comparaison.groupby('Party')[['Num_Women_2026', 'Num_Women_2024', 'diff']].sum().reset_index()
df_comparaison_women[df_comparaison_women["diff"]!=0]

,Party,Num_Women_2026,Num_Women_2024,diff
0,D,108.0,94.0,-14.0
2,R,17.0,31.0,14.0


In [18]:
df_comparaison_total2024 = df_comparaison.groupby(['State', 'Party'])[['Num_Women_2024', 'Total_2024']].sum().reset_index()

In [19]:
df_comparaison_total2024.groupby('Party')[['Num_Women_2024', 'Total_2024']].sum().assign(Pct_Women_2024=lambda x: (x['Num_Women_2024'] / x['Total_2024'] * 100).round(2)).reset_index()



,Party,Num_Women_2024,Total_2024,Pct_Women_2024
0,D,94.0,214.0,43.93
1,DEMOCRATIC-FARMER-LABOR,0.0,1.0,0.00
2,R,31.0,220.0,14.09


The total number of elected women to the House of Representatives didnt change between 2024 and 2026

In [ ]:
df_comparaison_total2024_state = df_comparaison_total2024.groupby('State')[['Num_Women_2024', 'Total_2024']].sum().assign(Pct_Women_2024=lambda x: (x['Num_Women_2024'] / x['Total_2024'] * 100).round(2)).reset_index().sort_values('Pct_Women_2024', ascending=False)


In [27]:
fig = px.choropleth(
    df_comparaison_total2024_state,
    locations='State',             
    locationmode='USA-states',
    color='Pct_Women_2024',
    scope='usa',
    color_continuous_scale='Purples',
    range_color=(0, 100),
    hover_data=['Num_Women_2024', 'Total_2024'],
    labels={'Pct_Women_2024': 'Femmes (%)'},
    title='Part des femmes élues à la Chambre des représentants en 2024, par État (%)'
)

fig.add_trace(go.Scattergeo(
    locations=df_comparaison_total2024_state['State'],
    locationmode='USA-states',
    text=df_comparaison_total2024_state['State'],
    mode='text',
    textfont={"size": 10, "color": 'black'},
    hoverinfo='skip',
    showlegend=False
))

fig.show()

In [22]:
df_comparaison_women = df_comparaison.groupby('State')[['Num_Women_2026', 'Num_Women_2024', 'diff']].sum().reset_index()

In [28]:
fig = px.choropleth(
    df_comparaison_women,
    locations='State',             
    locationmode='USA-states',
    color='diff',
    scope='usa',
    color_continuous_scale=['red', 'white', 'purple'],
    color_continuous_midpoint=0,
    range_color=(min(df_comparaison_women['diff']), max(df_comparaison_women['diff'])),
    hover_data=['Num_Women_2026', 'Num_Women_2024'],
    labels={'diff': 'Diference in Women 2026-2024'},
    title='Évolution du nombre de femmes élues à la Chambre des représentants entre 2024 et 2026, par État.',
    subtitle="Last update: 2026-11-05"
)

fig.add_trace(go.Scattergeo(
    locations=df_comparaison_women['State'],
    locationmode='USA-states',
    text=df_comparaison_women['State'],
    mode='text',
    textfont={"size": 10, "color": 'black'},
    hoverinfo='skip',
    showlegend=False
))

fig.show()